In [14]:
from datasets import Dataset
from transformers import (
 AutoTokenizer,
 AutoModelForSequenceClassification,
 TrainingArguments,
 Trainer,
)
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

In [15]:
MODEL_A = "distilbert-base-uncased"
LABELS = [
 "authentication", "network", "deployment", "database",
 "gpu", "api", "package", "general",
]
label2id = {label: i for i, label in enumerate(LABELS)}
id2label = {i: label for label, i in label2id.items()}
tokenizer_a = AutoTokenizer.from_pretrained(MODEL_A)
model_a = AutoModelForSequenceClassification.from_pretrained(
 MODEL_A,
 num_labels=len(LABELS),
 label2id=label2id,
 id2label=id2label,
)
def tokenize_a(batch):
 return tokenizer_a(batch["text"], truncation=True, max_length=128)
def compute_cls_metrics(eval_pred):
 logits, labels = eval_pred
 preds = np.argmax(logits, axis=-1)
 p, r, f1, _ = precision_recall_fscore_support(
 labels, preds, average="macro", zero_division=0
 )
 return {
 "accuracy": accuracy_score(labels, preds),
 "precision_macro": p,
 "recall_macro": r,
 "f1_macro": f1,
 }

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 6457.44it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [16]:
# TODO(student 1): load/construct at least 12 examples per intent,
# then create stratified train/validation/test splits.

examples_by_intent = {
    "authentication": [
        "I can't log in because my access token has expired.",
        "My password is correct, but the system says my credentials are invalid.",
        "How can I reset my forgotten password?",
        "The application rejects my JWT token as invalid.",
        "I am not receiving the two-factor authentication code.",
        "My session expires immediately after I sign in.",
        "How do I configure OAuth login for this application?",
        "My API key is rejected as unauthorized.",
        "My account was locked after several failed login attempts.",
        "Users cannot sign in through the company SSO provider.",
        "How can I refresh a token without making the user log in again?",
        "The login page keeps redirecting me back to the sign-in screen.",
    ],

    "network": [
        "The server cannot connect to the internet.",
        "Why does the connection time out when I contact the remote host?",
        "The domain name is not resolving to the correct IP address.",
        "I can ping the server locally but not from another machine.",
        "Which firewall port should I open for this service?",
        "The network connection becomes very slow during peak hours.",
        "My application cannot connect to a service on another subnet.",
        "DNS lookup fails inside the container.",
        "The server refuses every connection on port 8080.",
        "Packets are being dropped between the client and the server.",
        "The hostname works on my computer but not inside the virtual machine.",
        "How can I diagnose high network latency between two services?",
    ],

    "deployment": [
        "The latest release failed during deployment.",
        "How can I roll back to the previous application version?",
        "My CI pipeline succeeds, but the deployment step fails.",
        "The new container image is not being used after deployment.",
        "The application crashes after being deployed to production.",
        "How do I deploy this service to Kubernetes?",
        "The environment variables are missing in the production deployment.",
        "My Kubernetes rollout is stuck in progress.",
        "How can I perform a deployment without downtime?",
        "The staging build works, but the production build does not.",
        "The deployment pipeline cannot locate the generated artifact.",
        "How do I promote a release from staging to production?",
    ],

    "database": [
        "The application cannot connect to the database.",
        "My SQL query takes several minutes to complete.",
        "How can I create an index to improve query performance?",
        "The database migration failed halfway through.",
        "I am getting a deadlock error when updating records.",
        "How do I back up and restore the database?",
        "The database says the table does not exist.",
        "Some records are duplicated after inserting new data.",
        "The connection pool runs out of available connections.",
        "Database replication is falling behind the primary server.",
        "How can I change a column without losing existing data?",
        "The transaction is rolled back whenever I save the record.",
    ],

    "gpu": [
        "The program cannot detect my GPU.",
        "CUDA reports an out-of-memory error during training.",
        "The installed GPU driver is incompatible with CUDA.",
        "Why is the model training on the CPU instead of the GPU?",
        "GPU utilization remains at zero while the program is running.",
        "How can I select which GPU the application should use?",
        "The CUDA kernel fails when I start model training.",
        "My program works on one GPU but fails with multiple GPUs.",
        "The GPU temperature becomes too high during training.",
        "How can I reduce GPU memory usage for this model?",
        "PyTorch says that CUDA is not available.",
        "The system stopped recognizing the graphics card after an update.",
    ],

    "api": [
        "The API endpoint returns a 404 response.",
        "Why does this API request return an internal server error?",
        "How can I add pagination to this endpoint?",
        "The request body does not match the API schema.",
        "The API returns a rate-limit error after several requests.",
        "How do I send query parameters with this request?",
        "The endpoint returns an empty response instead of JSON.",
        "How can I document these endpoints using OpenAPI?",
        "The webhook is not receiving events from the service.",
        "The new API version breaks requests from older clients.",
        "How can I validate incoming data in the endpoint?",
        "The API accepts GET requests but rejects POST requests.",
    ],

    "package": [
        "Python says that the module cannot be found.",
        "Pip cannot install the package because of conflicting dependencies.",
        "How can I update this library to the latest version?",
        "The npm installation fails while resolving dependencies.",
        "Two installed packages require different versions of the same library.",
        "How do I remove an unused dependency from the project?",
        "The package works globally but not inside the virtual environment.",
        "The lock file contains a different package version than expected.",
        "How can I publish my Python package?",
        "The installed library does not contain the function from its documentation.",
        "Building the package fails because required metadata is missing.",
        "How do I install a specific version of this dependency?",
    ],

    "general": [
        "The application closes immediately after I open it.",
        "How can I enable debug logging?",
        "Where should I store the application's configuration settings?",
        "The program uses too much CPU while it is idle.",
        "How can I change the date format displayed by the application?",
        "The service produces no logs when it crashes.",
        "My configuration changes are ignored after restarting the program.",
        "How can I run this application in development mode?",
        "The server has run out of disk space.",
        "Where can I find the application's error logs?",
        "The program behaves differently on Windows and macOS.",
        "How can I reduce the application's startup time?",
    ],
}

examples = [
    {
        "text": text,
        "label": label2id[intent],
    }
    for intent, texts in examples_by_intent.items()
    for text in texts
]

In [17]:
from sklearn.model_selection import train_test_split
from transformers import DataCollatorWithPadding

# writing the text and labeles 
texts = [example["text"] for example in examples]
labels = [example["label"] for example in examples]

# Split each intent's 12 examples into 8 training and 4 for test and validation.
train_texts, temp_texts, train_labels, temp_labels = train_test_split(
    texts,
    labels,
    test_size=0.50,
    random_state=42,
    stratify=labels,
)

val_texts, test_texts, val_labels, test_labels = train_test_split(
    temp_texts,
    temp_labels,
    test_size=0.5,
    random_state=42,
    stratify=temp_labels,
)
# data collator for padding to make them as the same size
data_collator = DataCollatorWithPadding(tokenizer=tokenizer_a)

In [18]:
# setting up the dataset for tokenization

train_a = Dataset.from_dict({"text": train_texts, "label": train_labels})
val_a = Dataset.from_dict({"text": val_texts, "label": val_labels})
test_a = Dataset.from_dict({"text": test_texts, "label": test_labels})

train_a = train_a.map(tokenize_a, batched=True, remove_columns=["text"])
val_a = val_a.map(tokenize_a, batched=True, remove_columns=["text"])
test_a = test_a.map(tokenize_a, batched=True, remove_columns=["text"])

Map: 100%|██████████| 24/24 [00:00<00:00, 12953.71 examples/s]


In [19]:
args_a = TrainingArguments(
 output_dir="models/intent_classifier",
 learning_rate=2e-5,
 per_device_train_batch_size=8,
 per_device_eval_batch_size=8,
 num_train_epochs=3,
 eval_strategy="epoch",
 save_strategy="epoch",
 load_best_model_at_end=True,
 metric_for_best_model="f1_macro",
 greater_is_better=True,
 report_to="none",
 save_total_limit=1
)
trainer_a = Trainer(
 model=model_a,
 args=args_a,
 train_dataset=train_a,
 eval_dataset=val_a,
 compute_metrics=compute_cls_metrics,
 data_collator=data_collator, # for dynamic padding
)


In [20]:
# evaluation for baseline model
baseline_a = trainer_a.evaluate(test_a)

model_report_a = {
    "baseline": baseline_a,
    "fine_tuned": {},
    "quality_gate": {},
}

baseline_a

/Users/emad/Tuwaiq/Multi_Model_Agent_Architecture/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Accuracy,Precision Macro,Recall Macro,F1 Macro
No log,2.094641,0,0.083333,0.020979,0.083333,0.033482


{'eval_loss': 2.0946407318115234,
 'eval_accuracy': 0.08333333333333333,
 'eval_precision_macro': 0.02097902097902098,
 'eval_recall_macro': 0.08333333333333333,
 'eval_f1_macro': 0.033482142857142856}

In [21]:
trainer_a.train()
trainer_a.evaluate(test_a)
trainer_a.save_model("models/intent_classifier")
tokenizer_a.save_pretrained("models/intent_classifier")

/Users/emad/Tuwaiq/Multi_Model_Agent_Architecture/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,Precision Macro,Recall Macro,F1 Macro
1,No log,2.071703,0.125000,0.015625,0.125000,0.027778
2,No log,2.065389,0.125000,0.015625,0.125000,0.027778
3,No log,2.061971,0.125000,0.015625,0.125000,0.027778


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  8.84it/s]
/Users/emad/Tuwaiq/Multi_Model_Agent_Architecture/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  8.04it/s]
/Users/emad/Tuwaiq/Multi_Model_Agent_Architecture/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 11.21it/s]
/Users/emad/Tuwaiq/Multi_Model_Agent_Architecture/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Accuracy,Precision Macro,Recall Macro,F1 Macro
No log,2.084997,3,0.125000,0.015625,0.125000,0.027778


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  8.87it/s]


('models/intent_classifier/tokenizer_config.json',
 'models/intent_classifier/tokenizer.json')

In [30]:
import pandas as pd
history_df = pd.DataFrame(trainer_a.state.log_history)
train_logs = history_df[history_df["loss"].notna()].copy() if "loss" in history_df.columns else pd.DataFrame()
eval_logs = history_df[history_df["eval_loss"].notna()].copy() if "eval_loss" in history_df.columns else pd.DataFrame()

In [31]:
history_df

,eval_loss,eval_model_preparation_time,eval_accuracy,eval_precision_macro,eval_recall_macro,eval_f1_macro,eval_runtime,eval_samples_per_second,eval_steps_per_second,epoch,step,train_runtime,train_samples_per_second,train_steps_per_second,total_flos,train_loss
0,2.052474,0.0007,0.333333,0.229167,0.333333,0.241270,0.1946,123.299,15.412,1.0,6,NaN,NaN,NaN,NaN,NaN
1,2.035947,0.0007,0.291667,0.209091,0.291667,0.216071,0.2064,116.273,14.534,2.0,12,NaN,NaN,NaN,NaN,NaN
2,2.027932,0.0007,0.291667,0.212500,0.291667,0.220192,0.2084,115.166,14.396,3.0,18,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.0,18,6.1424,23.443,2.93,5.651167e+11,2.011741
4,2.071765,0.0007,0.166667,0.084559,0.166667,0.087500,0.1014,236.721,29.590,3.0,18,NaN,NaN,NaN,NaN,NaN


In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np

# Evaluate the fine-tuned model.
fine_tuned_a = trainer_a.evaluate(test_a)

# Generate predictions for the test set.
predictions_a = trainer_a.predict(test_a)
true_labels_a = predictions_a.label_ids
predicted_labels_a = np.argmax(
    predictions_a.predictions,
    axis=-1,
)

label_ids = list(range(len(LABELS)))

# Raw confusion-matrix counts.
confusion_counts_a = confusion_matrix(
    true_labels_a,
    predicted_labels_a,
    labels=label_ids,
)

confusion_counts_df = pd.DataFrame(
    confusion_counts_a,
    index=[f"Actual: {label}" for label in LABELS],
    columns=[f"Predicted: {label}" for label in LABELS],
)

# Per-class metrics.
per_class_report_a = classification_report(
    true_labels_a,
    predicted_labels_a,
    labels=label_ids,
    target_names=LABELS,
    output_dict=True,
    zero_division=0,
)

per_class_results_a = pd.DataFrame(
    per_class_report_a
).T

# Store Phase B results.
model_report_a["fine_tuned"] = fine_tuned_a
model_report_a["quality_gate"] = {
    "f1_improved": (
        fine_tuned_a["eval_f1_macro"]
        > baseline_a["eval_f1_macro"]
    )
}

print("Baseline metrics:")
display(pd.DataFrame([baseline_a]))

print("Fine-tuned metrics:")
display(pd.DataFrame([fine_tuned_a]))

print("Confusion matrix:")
display(confusion_counts_df)

print("Per-class metrics:")
display(per_class_results_a)

/Users/emad/Tuwaiq/Multi_Model_Agent_Architecture/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Accuracy,Precision Macro,Recall Macro,F1 Macro
No log,2.071765,3,0.166667,0.084559,0.166667,0.087500


/Users/emad/Tuwaiq/Multi_Model_Agent_Architecture/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Baseline metrics:


,eval_loss,eval_accuracy,eval_precision_macro,eval_recall_macro,eval_f1_macro
0,2.094641,0.083333,0.020979,0.083333,0.033482


...

,eval_loss,eval_accuracy,eval_precision_macro,eval_recall_macro,eval_f1_macro
0,2.071765,0.166667,0.084559,0.166667,0.0875


Confusion matrix:


,Predicted: authentication,Predicted: network,Predicted: deployment,Predicted: database,Predicted: gpu,Predicted: api,Predicted: package,Predicted: general
Actual: authentication,3,0,0,0,0,0,0,0
Actual: network,2,0,0,1,0,0,0,0
Actual: deployment,3,0,0,0,0,0,0,0
Actual: database,3,0,0,0,0,0,0,0
Actual: gpu,3,0,0,0,0,0,0,0
Actual: api,1,0,0,1,0,0,0,1
Actual: package,1,0,1,1,0,0,0,0
Actual: general,1,0,1,0,0,0,0,1


Per-class metrics:


,precision,recall,f1-score,support
authentication,0.176471,1.000000,0.300000,3.000000
network,0.000000,0.000000,0.000000,3.000000
deployment,0.000000,0.000000,0.000000,3.000000
database,0.000000,0.000000,0.000000,3.000000
gpu,0.000000,0.000000,0.000000,3.000000
api,0.000000,0.000000,0.000000,3.000000
package,0.000000,0.000000,0.000000,3.000000
general,0.500000,0.333333,0.400000,3.000000
accuracy,0.166667,0.166667,0.166667,0.166667
macro avg,0.084559,0.166667,0.087500,24.000000
